In [8]:
from chi import context, lease

# 1.1) lock in API version & select your project
context.version = "1.0"
context.choose_project()

# 1.2) pick the TACC endpoint
context.choose_site(default="CHI@TACC")

# 1.3) request one Cascade Lake R host
reservations = []
lease.add_node_reservation(
    reservations,
    node_type="compute_cascadelake_r",
    count=1
)

# 1.4) set duration → 24 h
start, end = lease.lease_duration(days=1)

# 1.5) create the lease
lease_resp = lease.create_lease(
    "coco-stage-lease",
    reservations=reservations,
    start_date=start,
    end_date=end
)
lease_id = lease_resp["id"]
print(f"Lease {lease_id} status: {lease_resp['status']} (pending)")

Lease a7aaa264-e0f2-4cb6-a72a-96dc750cbda5 status: PENDING (pending)


In [11]:
from chi import lease, server, context
import os, time

l = lease.get_lease("coco-stage-lease")
l.show()   # optional—prints status & reservations

# 3) Spawn a bare-metal VM on that host
username = os.getenv("USER")
s = server.Server(
    name=f"coco-stager-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04",
    flavor_name="baremetal"
)
s.submit(idempotent=True)
print("VM boot submitted; waiting for it to become ACTIVE…")


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>coco-stage-lea…

Lease Details:
Name: coco-stage-lease
ID: a7aaa264-e0f2-4cb6-a72a-96dc750cbda5
Status: ACTIVE
Start Date: 2025-05-06 04:06:00
End Date: 2025-05-07 04:05:00
User ID: e0c4fe1de33b696f0515f0d14443f20d8337d68360b4a354a79ae62eff9d113f
Project ID: d3c6e101843a4ba79e665ebf59b521a2

Node Reservations:
ID: d2a1ec7d-abf5-4c9b-a968-4db82c198157, Status: active, Min: 1, Max: 1

Floating IP Reservations:

Network Reservations:

Events:
Waiting for server coco-stager-xh2869_nyu_edu's status to become ACTIVE. This typically takes 10 minutes, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,coco-stager-xh2869_nyu_edu
Id,bb28db33-a196-42bc-9616-a6fd3536e5f2
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.0.113 (v4) Type: fixed MAC: bc:97:e1:c4:d1:b0
Network Name,sharednet1
Created At,2025-05-06T04:11:15Z
Keypair,xh2869_nyu_edu-jupyter
Reservation Id,d2a1ec7d-abf5-4c9b-a968-4db82c198157
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


VM boot submitted; waiting for it to become ACTIVE…


In [12]:
s.associate_floating_ip()

In [13]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.108.149 port 22.


Connection successful


In [14]:
s.refresh()
s.show(type="widget")

Attribute,coco-stager-xh2869_nyu_edu
Id,bb28db33-a196-42bc-9616-a6fd3536e5f2
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.0.113 (v4) Type: fixed MAC: bc:97:e1:c4:d1:b0 IP: 129.114.108.149 (v4) Type: floating MAC: bc:97:e1:c4:d1:b0
Network Name,sharednet1
Created At,2025-05-06T04:11:15Z
Keypair,xh2869_nyu_edu-jupyter
Reservation Id,d2a1ec7d-abf5-4c9b-a968-4db82c198157
Host Id,9acf860df16fe3cd915f9522cd52cf171577a815ef5c486f67a143e3


In [15]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

/opt/conda/lib/python3.10/site-packages/paramiko/client.py:889: UserWarning: Unknown ssh-ed25519 host key for 129.114.108.149: b'3599a233e948d3c89607bad485b0532b'
  warnings.warn(


# Executing docker install script, commit: 53a22f61c0628e58e1d6680b49e82993d304b449


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin >/dev/null

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c doc

Client: Docker Engine - Community
 Version:           28.1.1
 API version:       1.49
 Go version:        go1.23.8
 Git commit:        4eba377
 Built:             Fri Apr 18 09:52:14 2025
 OS/Arch:           linux/amd64
 Context:           default

Server: Docker Engine - Community
 Engine:
  Version:          28.1.1
  API version:      1.49 (minimum version 1.24)
  Go version:       go1.23.8
  Git commit:       01f442b
  Built:            Fri Apr 18 09:52:14 2025
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          1.7.27
  GitCommit:        05044ec0a9a75232cad458027ca83437aae3f4da
 runc:
  Version:          1.2.5
  GitCommit:        v1.2.5-0-g59923ef
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0


To run Docker as a non-privileged user, consider setting up the
Docker daemon in rootless mode for your user:

    dockerd-rootless-setuptool.sh install

Visit https://docs.docker.com/go/rootless/ to learn about rootless mode.


T

<Result cmd='sudo groupadd -f docker; sudo usermod -aG docker $USER' exited=0>

``` bash
# run on node-persist
curl https://rclone.org/install.sh | sudo bash
```

``` bash
# run on node-persist
# this line makes sure user_allow_other is un-commented in /etc/fuse.conf
sudo sed -i '/^#user_allow_other/s/^#//' /etc/fuse.conf
```

``` bash
# run on node-persist
mkdir -p ~/.config/rclone
nano  ~/.config/rclone/rclone.conf
```

Paste the following into the config file, but substitute your own application credential ID and secret.

You will also need to substitute your own user ID. You can find it using “Identity” \> “Users” in the Horizon GUI; it is an alphanumeric string (*not* the human-readable user name).

    [chi_tacc]
    type = swift
    user_id = YOUR_USER_ID
    application_credential_id = APP_CRED_ID
    application_credential_secret = APP_CRED_SECRET
    auth = https://chi.tacc.chameleoncloud.org:5000/v3
    region = CHI@TACC

Use Ctrl+O and Enter to save the file, and Ctrl+X to exit `nano`.

To test it, run

``` bash
# run on node-persist
rclone lsd chi_tacc:
```

### load data to object store (Please ignore if you're justing reading data)

``` bash
# run on coco-stager-xh2869
git clone https://github.com/kiritgpillai/Realtime-Traffic-Object-Detection-and-Tracking.git
```

``` bash
# run on coco-stager-xh2869
cd Real*
```

``` bash
# run on coco-stager-xh2869
checkout data
```

``` bash
# run on coco-stager-xh2869
cd ..
```

``` bash
# run on coco-stager-xh2869
docker compose -f ~/Realtime-Traffic-Object-Detection-and-Tracking/docker-compose-project[1].yaml run extract-data
```

``` bash
# run on coco-stager-xh2869
docker compose -f ~/Realtime-Traffic-Object-Detection-and-Tracking/docker-compose-project[1].yaml run transform-data
```

For the last stage, the container name is not specified in the Docker compose YAML - so we have to pass it as an environment variable first. Substitute your own container name in the line below:

``` bash
# run on coco-stager-xh2869
export RCLONE_CONTAINER=object-persist-project30
docker compose -f ~/Realtime-Traffic-Object-Detection-and-Tracking/docker-compose-project[1].yaml run load-data
```




### Mount your node to the object storage

``` bash
# run on your node
sudo mkdir -p /mnt/object
sudo chown -R cc /mnt/object
sudo chgrp -R cc /mnt/object
```

Now finally, we can use `rclone mount` to mount the object store at the mount point.

``` bash
# run on node-persist
rclone mount chi_tacc:object-persist-project30 /mnt/object --read-only --allow-other --daemon
```

Since we only intend to read the data, we can mount it in read-only mode and it will be slightly faster; and we are also protected from accidental writes. We also specified `--allow-other` so that we can use the mount from Docker, and `--daemon` means the `rclone` process will be started in the background.

Run

``` bash
# run on node-persist
ls /mnt/object
```

### Un-mount an object store

We’ll keep working with this object store in the next part, so you do not have to un-mount it now. But generally speaking to stop `rclone` running and un-mount the object store, you would run

    fusermount -u /mnt/object

where you specify the path of the mount point.

In [ ]:
# run in Chameleon Jupyter environment
from chi import server, context
import chi, os, time, datetime

context.version = "1.0" 
context.choose_project()
context.choose_site(default="CHI@TACC")

In [ ]:
username = os.getenv('USER') # all exp resources will have this prefix
s = server.get_server(f"coco-stager-{username}")
s.delete()